# 🔥 Análise de Incêndios no Brasil
**Zetta Lab — Desafio III** | Previsão e Severidade de Focos de Incêndio

Base: BDQueimadas/INPE — ~30M registros | PostgreSQL + PostGIS

## 0. Setup & Imports

In [ ]:
import sys, subprocess
pkgs = [
    'matplotlib', 'seaborn', 'plotly', 'folium',
    'scikit-learn', 'xgboost', 'joblib', 'pyarrow',
    'sqlalchemy', 'psycopg2-binary', 'requests', 'tqdm'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print('Dependencias OK!')

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from sqlalchemy import create_engine, text
from data_loader import get_engine, load_sample_from_db, get_db_stats, save_parquet, load_parquet
from features import (indice_angstrom, add_temporal_features, add_grid_cell,
                      encode_bioma, classificar_frp, classificar_angstrom)
from clustering import clusterizar_focos, resumo_clusters

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')
print('Setup concluido!')

## 1. Carregamento dos Dados

In [ ]:
stats = get_db_stats()
for k, v in stats.items():
    print(f'{k:25s}: {v}')

In [ ]:
PARQUET = '../csv/amostra_500k.parquet'

if os.path.exists(PARQUET):
    df = load_parquet(PARQUET)
    print(f'Parquet carregado: {len(df):,} registros')
else:
    print('Carregando amostra do banco...')
    df = load_sample_from_db(n=500_000)
    save_parquet(df, PARQUET)
    print(f'Amostra salva: {len(df):,} registros')

df['data_hora'] = pd.to_datetime(df['data_hora'])
df.head(3)

## 2. Limpeza e Pré-processamento

In [ ]:
print(f'Antes: {len(df):,} registros')
print('Nulos:\n', df.isnull().sum()[df.isnull().sum() > 0])

df = df.replace(-999, np.nan).replace(-999.0, np.nan)
df = df.dropna(subset=['latitude', 'longitude'])
df = df.drop_duplicates(subset=['id'])

print(f'Após limpeza: {len(df):,} registros')

In [ ]:
# DBSCAN clustering por janela semanal (raio 100km)
# Estratégia: clusterizar ANTES do enriquecimento para reduzir chamadas de API
from clustering import clusterizar_focos, resumo_clusters

print('Clusterizando focos por janela semanal (raio=100km)...')
df['data_hora'] = pd.to_datetime(df['data_hora'])
df['semana_cluster'] = df['data_hora'].dt.to_period('W').astype(str)

resultados_clust = []
for semana, grupo in df.groupby('semana_cluster'):
    if len(grupo) < 2:
        grupo['cluster_id'] = 0
        resultados_clust.append(grupo)
        continue
    g_c = clusterizar_focos(grupo, raio_km=100)
    g_c['semana_cluster'] = semana
    resultados_clust.append(g_c)

df_clust = pd.concat(resultados_clust, ignore_index=True)
res = resumo_clusters(df, df_clust)
print(f'Originais : {res["registros_originais"]:,}')
print(f'Clusters  : {res["registros_clusterizados"]:,}')
print(f'Reducao   : {res["reducao_pct"]:.1f}% — pontos unicos para enriquecer via API')

## 3. Schema PostgreSQL

Tabela `focos_incendio` no banco `incendios_db` — PostgreSQL 18 + PostGIS 3.6.

```sql
CREATE TABLE focos_incendio (
    id BIGINT PRIMARY KEY, data_hora TIMESTAMP,
    satelite VARCHAR(30), pais VARCHAR(30), estado VARCHAR(50),
    municipio VARCHAR(80), bioma VARCHAR(40),
    dias_sem_chuva INTEGER, precipitacao DOUBLE PRECISION,
    risco_fogo DOUBLE PRECISION, frp DOUBLE PRECISION,
    latitude DOUBLE PRECISION, longitude DOUBLE PRECISION,
    geom GEOMETRY(Point, 4326)
);
CREATE INDEX idx_focos_geom ON focos_incendio USING GIST(geom);
```

In [ ]:
# Verificar conexão e contagem real no banco
engine = get_engine()
with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM focos_incendio')).scalar()
    print(f'Registros no banco: {count:,}')

## 4. Enriquecimento com Dados Climáticos (NASA POWER API)

**Fluxo otimizado:** Clusterizamos primeiro (Fase 2) reduzindo os pontos únicos,
depois chamamos a NASA POWER API **apenas nos representantes de cluster**.
Isso reduz as chamadas de potencialmente milhões para alguns milhares.

A NASA POWER fornece dados históricos diários de temperatura (T2M) e umidade (RH2M)
com resolução de 0.5° × 0.5° (~55km), gratuita e sem chave de API.

In [ ]:
from climate_api import enriquecer_representantes, propagar_clima_para_df

# Os representantes de cluster já têm coordenadas únicas por região/semana
# Limitar a 500 representantes nesta demonstração (cada chamada leva ~0.3s)
LIMITE_DEMO = 500
df_repr = df_clust.head(LIMITE_DEMO).copy()

print(f'Enriquecendo {len(df_repr)} representantes via NASA POWER API...')
print('(Em producao: rodar sobre todos os {len(df_clust):,} representantes)')

df_repr_enriq = enriquecer_representantes(df_repr, delay_s=0.3, verbose=True)

print('\nAmostra enriquecida:')
df_repr_enriq[['latitude','longitude','temperatura','umidade','frp']].head(5)

In [ ]:
# Propagar temperatura/umidade para o df completo via cluster_id
# (todos os focos dentro do mesmo cluster 100km recebem o mesmo valor climático)
if 'cluster_id' in df.columns and 'cluster_id' in df_repr_enriq.columns:
    df = propagar_clima_para_df(df, df_repr_enriq, cluster_col='cluster_id')
else:
    # Fallback: merge direto pelos representantes enriquecidos
    df = df.merge(
        df_repr_enriq[['latitude','longitude','temperatura','umidade']].drop_duplicates(),
        on=['latitude','longitude'], how='left'
    )

cobertura = df['temperatura'].notna().mean() * 100
print(f'Cobertura climatica: {cobertura:.1f}% dos registros')
print(f'Temperatura media  : {df["temperatura"].mean():.1f} C')
print(f'Umidade media      : {df["umidade"].mean():.1f} %')

## 5. Feature Engineering

In [ ]:
df = add_temporal_features(df)
df = add_grid_cell(df, resolucao=0.5)
df['classe_frp'] = classificar_frp(df['frp'])

# Angstrom (apenas onde há umidade/temperatura)
mask = df['umidade'].notna() & df['temperatura'].notna()
df.loc[mask, 'angstrom'] = indice_angstrom(df.loc[mask,'umidade'], df.loc[mask,'temperatura'])

print('Distribuição classe_frp:')
print(df['classe_frp'].value_counts())
print('\nFeatures geradas:', list(df.columns))

## 6. Análise Exploratória (EDA)

In [ ]:
# Focos por bioma
fig = px.bar(
    df['bioma'].value_counts().reset_index(),
    x='bioma', y='count', title='Focos de Incêndio por Bioma',
    color='count', color_continuous_scale='Oranges',
    labels={'bioma':'Bioma','count':'Focos'}
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Sazonalidade mensal
por_mes = df.groupby('mes').size().reset_index(name='focos')
fig = px.bar(por_mes, x='mes', y='focos',
             title='Sazonalidade — Focos por Mês',
             labels={'mes':'Mês','focos':'Focos'},
             color='focos', color_continuous_scale='YlOrRd')
fig.show()

In [ ]:
# Top 10 estados
top_est = df['estado'].value_counts().head(10).reset_index()
fig = px.bar(top_est, x='estado', y='count',
             title='Top 10 Estados com Mais Focos',
             color='count', color_continuous_scale='Reds')
fig.show()

In [ ]:
# Mapa de densidade
df_mapa = df.dropna(subset=['latitude','longitude']).sample(min(50_000, len(df)), random_state=42)
fig = px.density_mapbox(
    df_mapa, lat='latitude', lon='longitude', z='risco_fogo',
    radius=4, zoom=3, center={'lat':-14,'lon':-52},
    mapbox_style='carto-positron',
    title='Mapa de Densidade — Focos de Incêndio',
    color_continuous_scale='YlOrRd'
)
fig.show()

In [ ]:
# Correlação entre variáveis numéricas
cols_corr = ['dias_sem_chuva','precipitacao','risco_fogo','frp','mes']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[cols_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlação entre Variáveis')
plt.tight_layout(); plt.show()

## 7. Modelo #1 — Previsão de Severidade (FRP)

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

df_m = df.dropna(subset=['frp']).copy()
le = LabelEncoder()
df_m['bioma_enc']  = le.fit_transform(df_m['bioma'].fillna('Desconhecido'))
df_m['estado_enc'] = le.fit_transform(df_m['estado'].fillna('Desconhecido'))

FEATS = ['latitude','longitude','dias_sem_chuva','precipitacao',
         'risco_fogo','mes','dia_do_ano','hora','bioma_enc','estado_enc']
X = df_m[FEATS].fillna(-1)
y = df_m['frp']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Treino: {len(X_tr):,} | Teste: {len(X_te):,}')

In [ ]:
modelo_sev = XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1
)
modelo_sev.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

y_pred = modelo_sev.predict(X_te)
print(f'MAE : {mean_absolute_error(y_te, y_pred):.2f} MW')
print(f'R²  : {r2_score(y_te, y_pred):.4f}')

In [ ]:
# Feature Importance
fi = pd.Series(modelo_sev.feature_importances_, index=FEATS).sort_values()
fig = px.bar(fi.reset_index(), x='feature_importances_', y='index',
             orientation='h', title='Feature Importance — Severidade FRP',
             color='feature_importances_', color_continuous_scale='Oranges',
             labels={'index':'Feature','feature_importances_':'Importância'})
fig.show()

In [ ]:
# Real vs Previsto
fig = px.scatter(x=y_te[:5000], y=y_pred[:5000],
                 labels={'x':'FRP Real (MW)','y':'FRP Previsto (MW)'},
                 title='Severidade: Real vs. Previsto', opacity=0.4,
                 color_discrete_sequence=['#FF6B35'])
lim = float(y_te.max())
fig.add_shape(type='line', x0=0, y0=0, x1=lim, y1=lim,
              line=dict(color='black', dash='dash'))
fig.show()

## 8. Modelo #2 — Previsão de Ocorrência Futura

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Grid 0.5° × 0.5° por semana
df_grid = add_grid_cell(df.copy())
df_grid['semana'] = df_grid['data_hora'].dt.to_period('W')

focos_celula = (
    df_grid.groupby(['cell_id','semana']).size()
    .reset_index(name='n_focos')
)
focos_celula['incendio'] = (focos_celula['n_focos'] > 0).astype(int)

stats_cel = df_grid.groupby('cell_id').agg(
    lat_grid=('lat_grid','first'), lon_grid=('lon_grid','first'),
    media_risco=('risco_fogo','mean'),
    media_dias_seco=('dias_sem_chuva','mean'),
    media_precip=('precipitacao','mean'),
    n_historico=('id','count')
).reset_index()

df_clf = focos_celula.merge(stats_cel, on='cell_id')
df_clf['mes'] = df_clf['semana'].dt.start_time.dt.month

pos = df_clf['incendio'].sum()
neg = (df_clf['incendio'] == 0).sum()
print(f'Positivos: {pos:,} | Negativos: {neg:,}')

In [ ]:
FEATS_C = ['lat_grid','lon_grid','media_risco','media_dias_seco',
           'media_precip','n_historico','mes']
X_c = df_clf[FEATS_C].fillna(-1)
y_c = df_clf['incendio']

X_tc, X_vc, y_tc, y_vc = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

modelo_prev = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=neg/pos if pos > 0 else 1,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
modelo_prev.fit(X_tc, y_tc, eval_set=[(X_vc, y_vc)], verbose=False)

y_prob = modelo_prev.predict_proba(X_vc)[:,1]
auc = roc_auc_score(y_vc, y_prob)
print(f'AUC-ROC: {auc:.4f}')
print(classification_report(y_vc, modelo_prev.predict(X_vc),
      target_names=['Sem Incêndio','Com Incêndio']))

In [ ]:
# Mapa de probabilidade futura
df_clf['prob_incendio'] = modelo_prev.predict_proba(X_c)[:,1]

fig = px.density_mapbox(
    df_clf, lat='lat_grid', lon='lon_grid', z='prob_incendio',
    radius=15, zoom=3, center={'lat':-14,'lon':-52},
    mapbox_style='carto-positron',
    color_continuous_scale='YlOrRd',
    title='Probabilidade de Ocorrência de Incêndio por Célula Geoespacial'
)
fig.show()

In [ ]:
# Curva ROC
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_vc, y_prob)
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
              name=f'ROC (AUC={auc:.3f})',
              line=dict(color='#FF6B35', width=2)))
fig.add_shape(type='line',x0=0,y0=0,x1=1,y1=1,
              line=dict(dash='dash',color='gray'))
fig.update_layout(title='Curva ROC — Previsão de Incêndio',
                  xaxis_title='Taxa de Falsos Positivos',
                  yaxis_title='Taxa de Verdadeiros Positivos')
fig.show()